# Week 4 Challenge

## Code commenter

In [7]:
import os
import sys
import gradio
import subprocess

from dotenv import load_dotenv
from google import generativeai
from openai import OpenAI

In [3]:

load_dotenv(override=True)
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
os.environ["GOOGLE_API_KEY"] = os.getenv('GOOGLE_API_KEY', 'your-key-if-not-using-env')

In [9]:
openai = OpenAI()
OPENAI_MODEL = "gpt-4o-mini"

GEMINI_MODEL = 'gemini-2.0-flash'
genai = generativeai
genai.configure()


In [41]:
system_message = "You are an assistant that adds docstrings and comments to Python code to maintain document readability"
system_message += "Respond with the full python code, only adding in docstrings and comments where neccessary."
system_message += "Comments should be included only where the intent of the code is unclear or refers to a complicated algorithm that may need to be referenced. You may also comment blocks of parameters that have a common purpose."
system_message += "Docstrings should be included for every function that is not named 'main'"
system_message += "Your reply should not include explanation of your reasoning outside of the comments and docstrings, nor should it alter the code otherwise."
system_message += "It is important that the code is unaltered as that could have significant business consequence."

In [13]:
def get_user_prompt(python):
    user_prompt = "Please add relevent comments to the follwoing python code: \n"
    user_prompt += python
    return user_prompt

In [15]:
def get_system_message(user_message):
    message = [
        {"role":"system", "content": system_message},
        {"role":"user", "content": get_user_prompt(user_message)}
    ]
    return message

In [37]:
def stream_gemini(python):
    model = genai.GenerativeModel(model_name=GEMINI_MODEL, system_instruction=system_message)
    response = model.generate_content(contents=get_user_prompt(python), stream=True)

    reply = ""
    for chunk in response:
        reply += chunk.text
        yield reply
   

In [46]:
def stream_gpt(python):
    system_instruct = get_system_message(python)
    stream = openai.chat.completions.create(model=OPENAI_MODEL, messages=system_instruct, stream=True)

    reply = ""
    for chunk in stream:
        reply += chunk.choices[0].delta.content or ""
        yield reply

In [22]:
def call_model(model_name, python):
    if model_name == GEMINI_MODEL:
        response = stream_gemini(python)
    if model_name == OPENAI_MODEL:
        response = stream_gpt(python)
    else:
        print("Error, unidentified model name")
    if response:
        yield from response
    else:
        return "Error - could not complete"

In [53]:
def read_file(file_name):
    with open(file_name, "r") as file:
        content = file.read()
    return content

def write_file(file_name, python):
    code = python.replace("```python","").replace("```","")
    with open(file_name, "w") as file:
        file.write(code)

In [51]:
def launch():
    with gradio.Blocks() as ui:
        gradio.Markdown("# Code Commenter")
        with gradio.Row():
            code_input = gradio.Textbox(label="Python Code", lines=10)
            code_output = gradio.Textbox(label="Commented Code", lines=10)
        with gradio.Row():
            file_input = gradio.Textbox(label="File Source")
            model = gradio.Dropdown([OPENAI_MODEL, GEMINI_MODEL], label="Select model", value=OPENAI_MODEL)
            run  = gradio.Button("Run")
        with gradio.Row():
            save = gradio.Button("Save")
            
        run.click(read_file, inputs=[file_input], outputs=[code_input]).then(
            call_model, inputs=[model, code_input], outputs=[code_output]
        )

        save.click(write_file, inputs=[file_input, code_output])
        
    ui.launch(inbrowser=True)

In [54]:
launch()

* Running on local URL:  http://127.0.0.1:7878
* To create a public link, set `share=True` in `launch()`.


Error, unidentified model name
